In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_DIR = '/content/drive/My Drive/covid_xray_data'
# Output: accuracy/loss curves and confusion matrices
FIG_OUTPUT_DIR = '/content/drive/My Drive/covid_results'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_OUTPUT_DIR, exist_ok=True)
%cd $DATA_DIR

def save_fig(name):
    """Save current figure to FIG_OUTPUT_DIR"""
    import matplotlib.pyplot as plt
    d = FIG_OUTPUT_DIR if 'FIG_OUTPUT_DIR' in dir() else './results'
    os.makedirs(d, exist_ok=True)
    plt.savefig(os.path.join(d, name), bbox_inches='tight', dpi=150)
    print('Saved:', os.path.join(d, name))

print('Data dir:', DATA_DIR)
print('Figures will be saved to:', FIG_OUTPUT_DIR)
if not os.path.isdir(os.path.join(DATA_DIR, 'Train')) or not os.path.isdir(os.path.join(DATA_DIR, 'Validation')):
    print('\n*** Upload your dataset: put Train/, Validation/, and Test/ folders inside', DATA_DIR)
    print('    (In Drive: My Drive → covid_xray_data → upload or create Train, Validation, Test)')

In [ ]:
ls

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.autograd import Variable
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
import copy
import cv2
import matplotlib.image as mpimg
import glob
from PIL import Image
from sklearn.metrics import f1_score

In [ ]:
try:
    data_dir = DATA_DIR
except NameError:
    data_dir = 'data'

In [ ]:
#Define transforms for the training data and testing data
train_transforms = transforms.Compose([transforms.RandomRotation(30),
                                       transforms.Resize(256),
                                       transforms.RandomResizedCrop(224),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor(),
                                       transforms.Normalize([0.485, 0.456, 0.406],
                                                            [0.229, 0.224, 0.225])])

test_transforms = transforms.Compose([transforms.Resize(256),
                                      transforms.CenterCrop(224),
                                      transforms.ToTensor(),
                                      transforms.Normalize([0.485, 0.456, 0.406],
                                                           [0.229, 0.224, 0.225])])

#pass transform here-in
train_data = datasets.ImageFolder(root='Train', transform=train_transforms)
# test_data = datasets.ImageFolder(root= 'test', transform=test_transforms)
valid_data = datasets.ImageFolder(root= 'Validation', transform=test_transforms)

print(len(valid_data))
print(len(train_data))
# print(len(test_data))

#data loaders
trainloader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True,num_workers=4)
# testloader = torch.utils.data.DataLoader(test_data, batch_size=64, shuffle=True,num_workers=4)
validloader = torch.utils.data.DataLoader(valid_data, batch_size=64, shuffle=True,num_workers=4)

print(len(trainloader))

print("Classes: ")
class_names = train_data.classes
print(class_names)

# **VGG16 Liklihood Loss**

In [ ]:
# Load the pretrained model from pytorch
vgg16 = models.vgg16(pretrained=True)
print(vgg16)
print('Output Layer of VGG16 : ', vgg16.classifier[6].out_features) # 1000

In [ ]:
features = list(vgg16.classifier.children())[:-7] # Remove last FC layers
features

In [ ]:
features.extend([nn.Linear(in_features=25088,out_features=630+100,bias=True)])
features.extend([nn.ReLU(inplace=True)])
features.extend([nn.Dropout(p=0.5,inplace=False)])
features.extend([nn.Linear(in_features=630+100,out_features=3,bias=False)])
features

In [ ]:
vgg16.classifier=nn.Sequential(*features)
vgg16

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()
# vg.train(mode=True)

In [ ]:
Epochs = 10
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(vgg16.parameters(), lr=0.001, momentum=0.9)

In [ ]:
for p in vgg16.parameters():
  print(p.requires_grad)

In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  vgg16.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = vgg16(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(vgg16,"vgg16_liklihood_loss.pth")

  vgg16.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = vgg16(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
save_fig('vgg16_focal_accuracy.png')  # save for sharing (Colab: in FIG_OUTPUT_DIR)
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
save_fig('vgg16_focal_loss.png')
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
save_fig('vgg16_focal_confusion_train.png')
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
save_fig('vgg16_focal_confusion_valid.png')
plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import multilabel_confusion_matrix
from sklearn.metrics import f1_score


vgg16=torch.load("vgg16_liklihood_loss.pth")
vgg16.eval()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

correct = 0
total = 0

running_loss=0.0

valid_p=[]
valid_l=[]
counter=0
with torch.no_grad():
    for data in validloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        label=labels
        labels= torch.nn.functional.one_hot(labels, len(class_names))
        for i in range(labels[:,0].shape[0]):
          if(labels[:,0][i]==1):
            labels[:,2][i]=1


        outputs = vgg16(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)

        limit = Variable(torch.Tensor([0.50])).to(device)
        predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


        loss = criterion(outputs, labels.float())

        correct += get_num_correct(predicted, labels)
        # print("Predicted",predicted)
        # print("Labels",labels)

        running_loss += loss.item()
        predicted= predicted.cpu().numpy()
        outputs=outputs.cpu().detach().numpy()

        for i in range(outputs.shape[0]):



          if (predicted[i]==[1,0,1]).all():
            valid_p.extend([0])
            counter+=1
          elif (predicted[i]==[0,1,0]).all():
            valid_p.extend([1])
          elif (predicted[i]==[0,0,1]).all():
            valid_p.extend([2])              # labels[:,2][i]=1
          else:
            l=outputs[i].argmax(axis=0)
            # print("Predicted",predicted[i])
            # print("label:",l)
            valid_p.extend([l])



        # valid_p.extend(predicted)
        valid_l.extend(label.cpu().numpy())
        # print(".",correct)
# valid_acc.append((100 * correct) / total)
# valid_loss.append(running_loss)
print(counter)
print(" ")


print("Validation Accuracy",((100 * correct) / total),"%")
print("Validation F1 Score :",f1_score(valid_l, valid_p, average="micro")) # p are predicted labels and l are true labels

valid_input=multilabel_confusion_matrix(valid_l,valid_p)

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[0],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 1")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[1],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 2")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[2],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 3")
plt.show()


# f1_score(labels.cpu().numpy(), predicted.cpu().numpy(), average="micro"))

In [ ]:
# Load the pretrained model from pytorch
vgg16 = models.vgg16(pretrained=True)
print(vgg16)
print('Output Layer of VGG16 : ', vgg16.classifier[6].out_features) # 1000

In [ ]:
features = list(vgg16.classifier.children())[:-7] # Remove last FC layers


features.extend([nn.Linear(in_features=25088,out_features=630+100,bias=True)])
features.extend([nn.ReLU(inplace=True)])
features.extend([nn.Dropout(p=0.5,inplace=False)])
features.extend([nn.Linear(in_features=630+100,out_features=3,bias=False)])
features

In [ ]:
vgg16.classifier=nn.Sequential(*features)
vgg16

Training loop

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()
# vg.train(mode=True)

In [ ]:
Epochs = 10
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(vgg16.parameters(), lr=0.01, momentum=0.9)

In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  vgg16.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = vgg16(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(vgg16,"vgg16_liklihood_loss_LR_0.01.pth")

  vgg16.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = vgg16(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import multilabel_confusion_matrix
from sklearn.metrics import f1_score


vgg16=torch.load("vgg16_liklihood_loss_LR_0.01.pth")
vgg16.eval()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

correct = 0
total = 0

running_loss=0.0

valid_p=[]
valid_l=[]
counter=0
with torch.no_grad():
    for data in validloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        label=labels
        labels= torch.nn.functional.one_hot(labels, len(class_names))
        for i in range(labels[:,0].shape[0]):
          if(labels[:,0][i]==1):
            labels[:,2][i]=1


        outputs = vgg16(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)

        limit = Variable(torch.Tensor([0.50])).to(device)
        predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


        loss = criterion(outputs, labels.float())

        correct += get_num_correct(predicted, labels)
        # print("Predicted",predicted)
        # print("Labels",labels)

        running_loss += loss.item()
        predicted= predicted.cpu().numpy()
        outputs=outputs.cpu().detach().numpy()

        for i in range(outputs.shape[0]):



          if (predicted[i]==[1,0,1]).all():
            valid_p.extend([0])
            counter+=1
          elif (predicted[i]==[0,1,0]).all():
            valid_p.extend([1])
          elif (predicted[i]==[0,0,1]).all():
            valid_p.extend([2])              # labels[:,2][i]=1
          else:
            l=outputs[i].argmax(axis=0)
            # print("Predicted",predicted[i])
            # print("label:",l)
            valid_p.extend([l])



        # valid_p.extend(predicted)
        valid_l.extend(label.cpu().numpy())
        # print(".",correct)
# valid_acc.append((100 * correct) / total)
# valid_loss.append(running_loss)
print(counter)
print(" ")


print("Validation Accuracy",((100 * correct) / total),"%")
print("Validation F1 Score :",f1_score(valid_l, valid_p, average="micro")) # p are predicted labels and l are true labels

valid_input=multilabel_confusion_matrix(valid_l,valid_p)

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[0],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 1")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[1],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 2")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[2],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 3")
plt.show()


# f1_score(labels.cpu().numpy(), predicted.cpu().numpy(), average="micro"))

# **Train Resnet-18**

In [ ]:
resnet18=models.resnet18(pretrained=True, progress=True)
print("hi")

features=[]
features.extend([nn.Linear(in_features=512,out_features=630+100,bias=True)])
features.extend([nn.ReLU(inplace=True)])
features.extend([nn.Dropout(p=0.5,inplace=False)])
features.extend([nn.Linear(in_features=630+100,out_features=3,bias=False)])
resnet18.fc=nn.Sequential(*features)
resnet18

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()
# vg.train(mode=True)

In [ ]:
Epochs = 10
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(resnet18.parameters(), lr=0.001, momentum=0.9)


In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  resnet18.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = resnet18(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(resnet18,"resnet18_liklihood_loss.pth")

  resnet18.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = resnet18(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Validation Confusion Matrix")
plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import multilabel_confusion_matrix
from sklearn.metrics import f1_score


resnet18=torch.load("resnet18_liklihood_loss.pth")
resnet18.eval()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18.to(device)

correct = 0
total = 0

running_loss=0.0

valid_p=[]
valid_l=[]
counter=0
with torch.no_grad():
    for data in validloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        label=labels
        labels= torch.nn.functional.one_hot(labels, len(class_names))
        for i in range(labels[:,0].shape[0]):
          if(labels[:,0][i]==1):
            labels[:,2][i]=1


        outputs = resnet18(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)

        limit = Variable(torch.Tensor([0.50])).to(device)
        predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


        loss = criterion(outputs, labels.float())

        correct += get_num_correct(predicted, labels)
        # print("Predicted",predicted)
        # print("Labels",labels)

        running_loss += loss.item()
        predicted= predicted.cpu().numpy()
        outputs=outputs.cpu().detach().numpy()

        for i in range(outputs.shape[0]):



          if (predicted[i]==[1,0,1]).all():
            valid_p.extend([0])
            counter+=1
          elif (predicted[i]==[0,1,0]).all():
            valid_p.extend([1])
          elif (predicted[i]==[0,0,1]).all():
            valid_p.extend([2])              # labels[:,2][i]=1
          else:
            l=outputs[i].argmax(axis=0)
            # print("Predicted",predicted[i])
            # print("label:",l)
            valid_p.extend([l])



        # valid_p.extend(predicted)
        valid_l.extend(label.cpu().numpy())
        # print(".",correct)
# valid_acc.append((100 * correct) / total)
# valid_loss.append(running_loss)
print(counter)
print(" ")


print("Validation Accuracy",((100 * correct) / total),"%")
print("Validation F1 Score :",f1_score(valid_l, valid_p, average="micro")) # p are predicted labels and l are true labels

valid_input=multilabel_confusion_matrix(valid_l,valid_p)

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[0],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 1")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[1],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 2")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[2],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 3")
plt.show()


# f1_score(labels.cpu().numpy(), predicted.cpu().numpy(), average="micro"))

In [ ]:
resnet18=models.resnet18(pretrained=True, progress=True)
print("hi")

features=[]
features.extend([nn.Linear(in_features=512,out_features=630+100,bias=True)])
features.extend([nn.ReLU(inplace=True)])
features.extend([nn.Dropout(p=0.5,inplace=False)])
features.extend([nn.Linear(in_features=630+100,out_features=3,bias=False)])
resnet18.fc=nn.Sequential(*features)
resnet18

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()
# vg.train(mode=True)

In [ ]:
Epochs = 10
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(resnet18.parameters(), lr=0.01, momentum=0.9)


In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  resnet18.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = resnet18(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(resnet18,"resnet18_liklihood_loss_LR_0.01.pth")

  resnet18.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = resnet18(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Validation Confusion Matrix")
plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import multilabel_confusion_matrix
from sklearn.metrics import f1_score


resnet18=torch.load("resnet18_liklihood_loss_LR_0.01.pth")
resnet18.eval()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18.to(device)

correct = 0
total = 0

running_loss=0.0

valid_p=[]
valid_l=[]
counter=0
with torch.no_grad():
    for data in validloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        label=labels
        labels= torch.nn.functional.one_hot(labels, len(class_names))
        for i in range(labels[:,0].shape[0]):
          if(labels[:,0][i]==1):
            labels[:,2][i]=1


        outputs = resnet18(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)

        limit = Variable(torch.Tensor([0.50])).to(device)
        predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


        loss = criterion(outputs, labels.float())

        correct += get_num_correct(predicted, labels)
        # print("Predicted",predicted)
        # print("Labels",labels)

        running_loss += loss.item()
        predicted= predicted.cpu().numpy()
        outputs=outputs.cpu().detach().numpy()

        for i in range(outputs.shape[0]):



          if (predicted[i]==[1,0,1]).all():
            valid_p.extend([0])
            counter+=1
          elif (predicted[i]==[0,1,0]).all():
            valid_p.extend([1])
          elif (predicted[i]==[0,0,1]).all():
            valid_p.extend([2])              # labels[:,2][i]=1
          else:
            l=outputs[i].argmax(axis=0)
            # print("Predicted",predicted[i])
            # print("label:",l)
            valid_p.extend([l])



        # valid_p.extend(predicted)
        valid_l.extend(label.cpu().numpy())
        # print(".",correct)
# valid_acc.append((100 * correct) / total)
# valid_loss.append(running_loss)
print(counter)
print(" ")


print("Validation Accuracy",((100 * correct) / total),"%")
print("Validation F1 Score :",f1_score(valid_l, valid_p, average="micro")) # p are predicted labels and l are true labels

valid_input=multilabel_confusion_matrix(valid_l,valid_p)

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[0],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 1")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[1],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 2")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[2],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 3")
plt.show()


# f1_score(labels.cpu().numpy(), predicted.cpu().numpy(), average="micro"))

# **( FOCAL LOSS )**

In [ ]:
class FocalLoss(nn.Module):

  def __init__(self,alpha,gamma):
    self.alpha=alpha
    self.gamma=gamma

    super().__init__()
    self.nll_loss = nn.BCEWithLogitsLoss()

  def forward(self,output,target):
    BCELoss=-(self.nll_loss(output,target))

    Pt=torch.exp(BCELoss)
    Standard_Focal_Loss=-((1-Pt)**self.gamma)*BCELoss
    Alpha_Balanced_Focal_Loss=self.alpha*Standard_Focal_Loss
    return Alpha_Balanced_Focal_Loss


## **Train VGG16** *Focal Loss*

In [ ]:
# Load the pretrained model from pytorch
vgg16 = models.vgg16(pretrained=True)
print(vgg16)
print('Output Layer of VGG16 : ', vgg16.classifier[6].out_features) # 1000

In [ ]:
features = list(vgg16.classifier.children())[:-7] # Remove last FC layers
features

In [ ]:
features.extend([nn.Linear(in_features=25088,out_features=630+100,bias=True)])
features.extend([nn.ReLU(inplace=True)])
features.extend([nn.Dropout(p=0.5,inplace=False)])
features.extend([nn.Linear(in_features=630+100,out_features=3,bias=False)])
features

In [ ]:
vgg16.classifier=nn.Sequential(*features)
vgg16

# **Training Loop (Alpha=0.5, Gamma=2, LR=0.001)**


In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()
# vg.train(mode=True)

Epochs = 10
# criterion = nn.BCEWithLogitsLoss()
criterion=FocalLoss(alpha=0.5,gamma=2)
optimizer = optim.SGD(vgg16.parameters(), lr=0.001, momentum=0.9)

In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  vgg16.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = vgg16(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(vgg16,"vgg16_focal_loss.pth")

  vgg16.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = vgg16(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

# **Training Loop (Alpha=2, Gamma=1, LR=0.001)**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()
# vg.train(mode=True)

Epochs = 10
# criterion = nn.BCEWithLogitsLoss()
criterion=FocalLoss(alpha=2,gamma=1)
optimizer = optim.SGD(vgg16.parameters(), lr=0.001, momentum=0.9)

In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  vgg16.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = vgg16(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(vgg16,"vgg16_focal_loss_exp2.pth")

  vgg16.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = vgg16(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

# **Training Loop (Alpha=2, Gamma=1, LR=0.01)**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()
# vg.train(mode=True)

Epochs = 10
# criterion = nn.BCEWithLogitsLoss()
criterion=FocalLoss(alpha=2,gamma=1)
optimizer = optim.SGD(vgg16.parameters(), lr=0.01, momentum=0.9)

In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  vgg16.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = vgg16(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(vgg16,"vgg16_focal_loss_exp3.pth")

  vgg16.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = vgg16(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

# **Training Loop (Alpha=1.5, Gamma=1, LR=0.001)**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()
# vg.train(mode=True)

Epochs = 10
# criterion = nn.BCEWithLogitsLoss()
criterion=FocalLoss(alpha=1.5,gamma=1)
optimizer = optim.SGD(vgg16.parameters(), lr=0.001, momentum=0.9)

In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  vgg16.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = vgg16(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(vgg16,"vgg16_focal_loss_exp4.pth")

  vgg16.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = vgg16(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

# **Training Loop (Alpha=1.5, Gamma=0.5, LR=0.001) BEST MODEL o_o**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()
# vg.train(mode=True)

Epochs = 10
# criterion = nn.BCEWithLogitsLoss()
criterion=FocalLoss(alpha=1.5,gamma=0.5)
optimizer = optim.SGD(vgg16.parameters(), lr=0.001, momentum=0.9)

In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  vgg16.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = vgg16(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(vgg16,"vgg16_focal_loss_exp5.pth")

  vgg16.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = vgg16(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import multilabel_confusion_matrix
from sklearn.metrics import f1_score


vgg16=torch.load("vgg16_focal_loss_exp5.pth")
vgg16.eval()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

correct = 0
total = 0

running_loss=0.0

valid_p=[]
valid_l=[]
counter=0
with torch.no_grad():
    for data in validloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        label=labels
        labels= torch.nn.functional.one_hot(labels, len(class_names))
        for i in range(labels[:,0].shape[0]):
          if(labels[:,0][i]==1):
            labels[:,2][i]=1


        outputs = vgg16(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)

        limit = Variable(torch.Tensor([0.50])).to(device)
        predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


        loss = criterion(outputs, labels.float())

        correct += get_num_correct(predicted, labels)
        # print("Predicted",predicted)
        # print("Labels",labels)

        running_loss += loss.item()
        predicted= predicted.cpu().numpy()
        outputs=outputs.cpu().detach().numpy()

        for i in range(outputs.shape[0]):



          if (predicted[i]==[1,0,1]).all():
            valid_p.extend([0])
            counter+=1
          elif (predicted[i]==[0,1,0]).all():
            valid_p.extend([1])
          elif (predicted[i]==[0,0,1]).all():
            valid_p.extend([2])              # labels[:,2][i]=1
          else:
            l=outputs[i].argmax(axis=0)
            # print("Predicted",predicted[i])
            # print("label:",l)
            valid_p.extend([l])



        # valid_p.extend(predicted)
        valid_l.extend(label.cpu().numpy())
        # print(".",correct)
# valid_acc.append((100 * correct) / total)
# valid_loss.append(running_loss)
print(counter)
print(" ")


print("Validation Accuracy",((100 * correct) / total),"%")
print("Validation F1 Score :",f1_score(valid_l, valid_p, average="micro")) # p are predicted labels and l are true labels

valid_input=multilabel_confusion_matrix(valid_l,valid_p)

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[0],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 1")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[1],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 2")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[2],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 3")
plt.show()


# f1_score(labels.cpu().numpy(), predicted.cpu().numpy(), average="micro"))

# **Training Loop (Alpha=1.5, Gamma=0.25, LR=0.001)**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()
# vg.train(mode=True)

Epochs = 10
# criterion = nn.BCEWithLogitsLoss()
criterion=FocalLoss(alpha=1.5,gamma=0.25)
optimizer = optim.SGD(vgg16.parameters(), lr=0.001, momentum=0.9)

In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  vgg16.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = vgg16(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(vgg16,"vgg16_focal_loss_exp6.pth")

  vgg16.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = vgg16(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

# **Training Loop (Alpha=1.5, Gamma=0.25, LR=0.01)**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()
# vg.train(mode=True)

Epochs = 10
# criterion = nn.BCEWithLogitsLoss()
criterion=FocalLoss(alpha=1.5,gamma=0.25)
optimizer = optim.SGD(vgg16.parameters(), lr=0.01, momentum=0.9)

In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  vgg16.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = vgg16(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(vgg16,"vgg16_focal_loss_exp7.pth")

  vgg16.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = vgg16(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

# **Training Loop (Alpha=2.5, Gamma=0.25, LR=0.001)**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()
# vg.train(mode=True)

Epochs = 10
# criterion = nn.BCEWithLogitsLoss()
criterion=FocalLoss(alpha=2.5,gamma=0.25)
optimizer = optim.SGD(vgg16.parameters(), lr=0.001, momentum=0.9)

In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  vgg16.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = vgg16(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(vgg16,"vgg16_focal_loss_exp8.pth")

  vgg16.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = vgg16(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

# **Training Loop (Alpha=2, Gamma=0.5, LR=0.001)**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()
# vg.train(mode=True)

Epochs = 10
# criterion = nn.BCEWithLogitsLoss()
criterion=FocalLoss(alpha=2,gamma=0.5)
optimizer = optim.SGD(vgg16.parameters(), lr=0.001, momentum=0.9)

In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
vgg16.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  vgg16.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = vgg16(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(vgg16,"vgg16_focal_loss_exp9.pth")

  vgg16.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = vgg16(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

# **Train Resnet18** Focal Loss

In [ ]:
resnet18=models.resnet18(pretrained=True, progress=True)
print("hi")

features=[]
features.extend([nn.Linear(in_features=512,out_features=630+100,bias=True)])
features.extend([nn.ReLU(inplace=True)])
features.extend([nn.Dropout(p=0.5,inplace=False)])
features.extend([nn.Linear(in_features=630+100,out_features=3,bias=False)])
resnet18.fc=nn.Sequential(*features)
resnet18

# **Training Loop (Alpha=0.5, Gamma=2, LR=0.001)**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()


Epochs = 10
criterion = FocalLoss(alpha=0.5,gamma=2)
optimizer = optim.SGD(resnet18.parameters(), lr=0.001, momentum=0.9)


In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  resnet18.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = resnet18(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(resnet18,"‘res18_focal_loss.pth")

  resnet18.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = resnet18(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Validation Confusion Matrix")
plt.show()

from sklearn.metrics import f1_score
print("Training   F1 Score :",f1_score(train_l, train_p)) # p are predicted labels and l are true labels
print("Validation F1 Score :",f1_score(valid_l, valid_p)) # p are predicted labels and l are true labels


# **Training Loop (Alpha=2, Gamma=1, LR=0.001)**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()


Epochs = 10
criterion = FocalLoss(alpha=2,gamma=1)
optimizer = optim.SGD(resnet18.parameters(), lr=0.001, momentum=0.9)


In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  resnet18.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = resnet18(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(resnet18,"‘res18_focal_loss_exp2.pth")

  resnet18.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = resnet18(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Validation Confusion Matrix")
plt.show()

# from sklearn.metrics import f1_score
# print("Training   F1 Score :",f1_score(train_l, train_p)) # p are predicted labels and l are true labels
# print("Validation F1 Score :",f1_score(valid_l, valid_p)) # p are predicted labels and l are true labels


# **Training Loop (Alpha=2, Gamma=1, LR=0.01) BEST MODEL 0_0**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()


Epochs = 10
criterion = FocalLoss(alpha=2,gamma=1)
optimizer = optim.SGD(resnet18.parameters(), lr=0.01, momentum=0.9)


In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  resnet18.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = resnet18(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(resnet18,"res18_focal_loss_exp3.pth")

  resnet18.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = resnet18(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Validation Confusion Matrix")
plt.show()

# from sklearn.metrics import f1_score
# print("Training   F1 Score :",f1_score(train_l, train_p)) # p are predicted labels and l are true labels
# print("Validation F1 Score :",f1_score(valid_l, valid_p)) # p are predicted labels and l are true labels


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import multilabel_confusion_matrix
from sklearn.metrics import f1_score


resnet18=torch.load("res18_focal_loss_exp3.pth")
resnet18.eval()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18.to(device)

correct = 0
total = 0

running_loss=0.0

valid_p=[]
valid_l=[]
counter=0
with torch.no_grad():
    for data in validloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        label=labels
        labels= torch.nn.functional.one_hot(labels, len(class_names))
        for i in range(labels[:,0].shape[0]):
          if(labels[:,0][i]==1):
            labels[:,2][i]=1


        outputs = resnet18(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)

        limit = Variable(torch.Tensor([0.50])).to(device)
        predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


        loss = criterion(outputs, labels.float())

        correct += get_num_correct(predicted, labels)
        # print("Predicted",predicted)
        # print("Labels",labels)

        running_loss += loss.item()
        predicted= predicted.cpu().numpy()
        outputs=outputs.cpu().detach().numpy()

        for i in range(outputs.shape[0]):



          if (predicted[i]==[1,0,1]).all():
            valid_p.extend([0])
            counter+=1
          elif (predicted[i]==[0,1,0]).all():
            valid_p.extend([1])
          elif (predicted[i]==[0,0,1]).all():
            valid_p.extend([2])              # labels[:,2][i]=1
          else:
            l=outputs[i].argmax(axis=0)
            # print("Predicted",predicted[i])
            # print("label:",l)
            valid_p.extend([l])



        # valid_p.extend(predicted)
        valid_l.extend(label.cpu().numpy())
        # print(".",correct)
# valid_acc.append((100 * correct) / total)
# valid_loss.append(running_loss)
print(counter)
print(" ")


print("Validation Accuracy",((100 * correct) / total),"%")
print("Validation F1 Score :",f1_score(valid_l, valid_p, average="micro")) # p are predicted labels and l are true labels

valid_input=multilabel_confusion_matrix(valid_l,valid_p)

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[0],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 1")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[1],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 2")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input[2],cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Testing Confusion Matrix 3")
plt.show()


# f1_score(labels.cpu().numpy(), predicted.cpu().numpy(), average="micro"))

# **Training Loop (Alpha=1.5, Gamma=1, LR=0.001)**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()


Epochs = 10
criterion = FocalLoss(alpha=1.5,gamma=1)
optimizer = optim.SGD(resnet18.parameters(), lr=0.001, momentum=0.9)


In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  resnet18.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = resnet18(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(resnet18,"‘res18_focal_loss_exp4.pth")

  resnet18.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = resnet18(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Validation Confusion Matrix")
plt.show()

# from sklearn.metrics import f1_score
# print("Training   F1 Score :",f1_score(train_l, train_p)) # p are predicted labels and l are true labels
# print("Validation F1 Score :",f1_score(valid_l, valid_p)) # p are predicted labels and l are true labels


# **Training Loop (Alpha=1.5, Gamma=0.5, LR=0.001)**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()


Epochs = 10
criterion = FocalLoss(alpha=1.5,gamma=0.5)
optimizer = optim.SGD(resnet18.parameters(), lr=0.001, momentum=0.9)


In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  resnet18.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = resnet18(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(resnet18,"‘res18_focal_loss_exp5.pth")

  resnet18.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = resnet18(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Validation Confusion Matrix")
plt.show()

# from sklearn.metrics import f1_score
# print("Training   F1 Score :",f1_score(train_l, train_p)) # p are predicted labels and l are true labels
# print("Validation F1 Score :",f1_score(valid_l, valid_p)) # p are predicted labels and l are true labels


# **Training Loop (Alpha=1.5, Gamma=0.25, LR=0.001)**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()


Epochs = 10
criterion = FocalLoss(alpha=1.5,gamma=0.25)
optimizer = optim.SGD(resnet18.parameters(), lr=0.001, momentum=0.9)


In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  resnet18.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = resnet18(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(resnet18,"‘res18_focal_loss_exp6.pth")

  resnet18.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = resnet18(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Validation Confusion Matrix")
plt.show()

# from sklearn.metrics import f1_score
# print("Training   F1 Score :",f1_score(train_l, train_p)) # p are predicted labels and l are true labels
# print("Validation F1 Score :",f1_score(valid_l, valid_p)) # p are predicted labels and l are true labels


# **Training Loop (Alpha=0.5, Gamma=4, LR=0.001)**

In [ ]:
def get_num_correct(preds, labs):
    # return preds.argmax(dim=1).eq(labels).sum().item()
    return torch.sum(torch.all(torch.eq(preds, labs),dim=1)).item()


Epochs = 10
criterion = FocalLoss(alpha=0.5,gamma=4)
optimizer = optim.SGD(resnet18.parameters(), lr=0.001, momentum=0.9)


In [ ]:
from tqdm import tqdm

#if you have gpu then you need to convert the network and data to cuda
#the easiest way is to first check for device and then convert network and data to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18.to(device)

train_acc=[]
valid_acc=[]

train_loss=[]
valid_loss=[]

for epoch in range(Epochs):  # loop over the dataset multiple times

  resnet18.train()
  running_loss = 0.0
  total_correct=0.0
  counter=0
  #
  train_p=[]
  train_l=[]
  pbar = tqdm(enumerate(trainloader))
  for i, data in pbar:


    # get the inputs
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    label=labels
    labels= torch.nn.functional.one_hot(labels, len(class_names))
    for i in range(labels[:,0].shape[0]):
      if(labels[:,0][i]==1):
        labels[:,2][i]=1

    # zero the parameter gradients
    optimizer.zero_grad()
    # In PyTorch, we need to set the gradients to zero before starting to do backpropragation
    # because PyTorch accumulates the gradients on subsequent backward passes.
    # This is convenient while training RNNs.
    # So, the default action is to accumulate the gradients on every loss.backward() call

    # forward + backward + optimize
    outputs = resnet18(inputs)               #----> forward pass
    # _, predict = torch.max(outputs.data, 1)

    loss = criterion(outputs, labels.float())   #----> compute loss
    loss.backward()                     #----> backward pass
    optimizer.step()                    #----> weights update

    # print statistics
    running_loss += loss.item()

    limit = Variable(torch.Tensor([0.50])).to(device)
    # # print("Before",outputs)
    # # outputs=torch.Sigmoid(outputs)
    # o=outputs.cpu()
    # o=o.detach().numpy()
    # outputs = 1/(1 + np.exp(-o))
    # # print("Afterr",outputs)

    predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1
    # print("labeleddd", labels)
    # print("outputedd",outputs)
    # print("predicted", predicted)

    # total_correct += torch.sum(torch.all(torch.eq(predicted, labels),dim=1)).item()
    total_correct += get_num_correct(predicted, labels)

    predicted= predicted.cpu().numpy()
    outputs= outputs.cpu().detach().numpy()
    # print("After",predicted)

    for i in range(outputs.shape[0]):



      if (predicted[i]==[1,0,1]).all():
        train_p.extend([0])
        counter+=1
      elif (predicted[i]==[0,1,0]).all():
        train_p.extend([1])
      elif (predicted[i]==[0,0,1]).all():
         train_p.extend([2])
      else:
          l=outputs[i].argmax(axis=0)
          # print("Predicted",predicted[i])
          # print("label:",l)
          train_p.extend([l])





    # train_p.extend(predicted)
    train_l.extend(label.cpu().numpy())

    # print("train predicted",train_p)

    # print("train label",train_l)

    pbar.set_description(

            'Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, i * len(inputs), len(trainloader.dataset),
                100. * i / len(trainloader),
                loss.data.item()))


  a=(total_correct*100)/len(train_data)
  train_acc.append(a)
  train_loss.append(running_loss)
  print("Train:","epoch: ", epoch, "total_correct: ", total_correct, "total_loss: ", running_loss,"counter:",counter)
  torch.save(resnet18,"‘res18_focal_loss_exp.pth")

  resnet18.eval()
  from sklearn.metrics import confusion_matrix
  from sklearn.metrics import multilabel_confusion_matrix
  correct = 0
  total = 0

  running_loss=0.0

  valid_p=[]
  valid_l=[]
  counter=0
  with torch.no_grad():
      for data in validloader:
          images, labels = data
          images, labels = images.to(device), labels.to(device)

          label=labels
          labels= torch.nn.functional.one_hot(labels, len(class_names))
          for i in range(labels[:,0].shape[0]):
            if(labels[:,0][i]==1):
              labels[:,2][i]=1


          outputs = resnet18(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)

          limit = Variable(torch.Tensor([0.50])).to(device)
          predicted = torch.where(outputs < limit, Variable(torch.Tensor([0])).to(device), Variable(torch.Tensor([1])).to(device)) # produce labels 0 and 1


          loss = criterion(outputs, labels.float())

          correct += get_num_correct(predicted, labels)
          # print("Predicted",predicted)
          # print("Labels",labels)

          running_loss += loss.item()
          predicted= predicted.cpu().numpy()
          outputs=outputs.cpu().detach().numpy()

          for i in range(outputs.shape[0]):



            if (predicted[i]==[1,0,1]).all():
              valid_p.extend([0])
              counter+=1
            elif (predicted[i]==[0,1,0]).all():
              valid_p.extend([1])
            elif (predicted[i]==[0,0,1]).all():
              valid_p.extend([2])              # labels[:,2][i]=1
            else:
              l=outputs[i].argmax(axis=0)
              # print("Predicted",predicted[i])
              # print("label:",l)
              valid_p.extend([l])



          # valid_p.extend(predicted)
          valid_l.extend(label.cpu().numpy())
          # print(".",correct)
  valid_acc.append((100 * correct) / total)
  valid_loss.append(running_loss)
  print("Valid:","epoch: ", epoch, "total_correct: ", correct, "total_loss: ", running_loss,"counter:",counter)
  print(" ")

print("Training Accuracy",a,"%")

print("Validation Accuracy",((100 * correct) / total),"%")

print('Finished Training')



plt.plot(train_acc, color="b")
plt.plot(valid_acc, color="r")
plt.title("Accuracy Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy %")
plt.show()

plt.plot(train_loss, color="b")
plt.plot(valid_loss, color="r")
plt.title("Loss Curve of Training and Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss ")
plt.show()

train_input=confusion_matrix(train_p,train_l)
valid_input=confusion_matrix(valid_p,valid_l)
print(train_input.shape)
import seaborn as sns
plt.subplots(figsize=(4,4))
ax = sns.heatmap(train_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Training Confusion Matrix")
plt.show()

plt.subplots(figsize=(4,4))
ax = sns.heatmap(valid_input,cmap="Blues", annot=True, fmt="d",linewidths=1.0,cbar_kws={"shrink": .80})
ax.set_xlabel('Predicted');
ax.set_ylabel('True');
plt.title("Validation Confusion Matrix")
plt.show()

# from sklearn.metrics import f1_score
# print("Training   F1 Score :",f1_score(train_l, train_p)) # p are predicted labels and l are true labels
# print("Validation F1 Score :",f1_score(valid_l, valid_p)) # p are predicted labels and l are true labels


# ***Load Testing Data and Test***

In [ ]:
test_transforms = transforms.Compose([transforms.Resize(256),
                                      transforms.CenterCrop(224),
                                      transforms.ToTensor(),
                                      transforms.Normalize([0.485, 0.456, 0.406],
                                                           [0.229, 0.224, 0.225])])

class ImageFolderWithPaths(torchvision.datasets.ImageFolder):
    def __getitem__(self, index):
        img_lab = super(ImageFolderWithPaths, self).__getitem__(index)
        path = self.imgs[index][0]
        img_lab_path = (img_lab + (path,))
        return img_lab_path

test_data = ImageFolderWithPaths(root='Test', transform=test_transforms)
testloader = torch.utils.data.DataLoader(dataset=test_data,batch_size=64,shuffle=True,num_workers=4)
print("Total Test Images:",len(test_data))
len(testloader)